# Performance & Cost + Enterprise Architecture for Telecom AI
## LangGraph + LangSmith Concepts + Milvus / Zilliz Cloud Concepts + MCP
### Beginner-friendly step-by-step hands-on notebook

This notebook is a practical lab for understanding how to design a **telecom AI workflow** that balances:

- performance
- cost
- token usage
- caching
- rate limiting
- model selection
- enterprise architecture
- planner / executor / validator pattern
- multi-agent telecom workflows
- OSS/BSS integration

---

## What you will build

A small telecom workflow for a request like:

> "My mobile internet is slow in Pune. Diagnose the likely issue and recommend next action."

The workflow will:

1. plan the steps
2. retrieve telecom guidance from a Milvus / Zilliz-like store
3. execute retrieval and diagnosis
4. validate the answer
5. estimate token usage and cost
6. show where caching and rate limiting help
7. explain how this maps to enterprise telecom architecture

## Why these technologies matter

### LangGraph
LangGraph organizes workflows with **state, nodes, and edges**. It is a good fit for Planner–Executor–Validator designs.

### LangSmith
LangSmith helps with observability:
- traces
- runs
- projects
- debugging
- cost and token inspection

### MCP
MCP provides a standard way to think about:
- tools
- resources
- prompts

### Zilliz Cloud / Milvus
Zilliz Cloud is a managed Milvus service for vector retrieval. In this notebook we use a mock vector store so the lab stays easy to run.

In [ ]:
# Uncomment if needed in a fresh environment
# %pip install -U langgraph pandas

## Step 1 — Imports

In [1]:
from __future__ import annotations

from typing import TypedDict, Dict, Any, List, Optional
from pprint import pprint
import hashlib
import pandas as pd

from langgraph.graph import StateGraph, END

## Step 2 — Beginner concepts

### Token optimization
Reduce unnecessary text sent to the model.

### Model selection
Choose the right model for the task:
- smaller/cheaper model for routing or validation
- stronger model for generation only when needed

### Caching
Reuse previous results instead of recomputing the same answer.

### Rate limiting
Protect external systems and model APIs from too many requests.

### Planner – Executor – Validator
A common enterprise pattern:
- Planner decides what to do
- Executor performs the work
- Validator checks if the output is acceptable

## Step 3 — MCP-style telecom catalog

In [2]:
mcp_catalog = {
    "tools": [
        {"name": "get_customer_profile", "description": "Fetch telecom customer profile"},
        {"name": "check_outage", "description": "Check outage for city and issue type"},
        {"name": "search_vector_store", "description": "Search telecom SOP/RCA guidance"},
        {"name": "create_ticket", "description": "Create support ticket in OSS/BSS"},
    ],
    "resources": [
        {"name": "telecom_guidance_store", "description": "Telecom SOP and RCA knowledge base"},
    ],
    "prompts": [
        {"name": "telecom_diagnosis_prompt", "description": "Prompt template for telecom diagnosis"},
    ],
}

pprint(mcp_catalog)

{'prompts': [{'description': 'Prompt template for telecom diagnosis',
              'name': 'telecom_diagnosis_prompt'}],
 'resources': [{'description': 'Telecom SOP and RCA knowledge base',
                'name': 'telecom_guidance_store'}],
 'tools': [{'description': 'Fetch telecom customer profile',
            'name': 'get_customer_profile'},
           {'description': 'Check outage for city and issue type',
            'name': 'check_outage'},
           {'description': 'Search telecom SOP/RCA guidance',
            'name': 'search_vector_store'},
           {'description': 'Create support ticket in OSS/BSS',
            'name': 'create_ticket'}]}


## Step 4 — Mock telecom data

In [3]:
CUSTOMERS = {
    "CUST_1001": {
        "customer_id": "CUST_1001",
        "name": "Ravi Patil",
        "city": "Pune",
        "plan": "Premium 5G",
        "network_type": "5G",
        "recent_complaints": 2,
    },
    "CUST_2001": {
        "customer_id": "CUST_2001",
        "name": "Meera Shah",
        "city": "Mumbai",
        "plan": "Standard 4G",
        "network_type": "4G",
        "recent_complaints": 0,
    },
}

OUTAGES = [
    {"city": "Pune", "issue_type": "slow_internet", "outage_detected": False, "summary": "No citywide outage found."},
    {"city": "Mumbai", "issue_type": "packet_loss", "outage_detected": True, "summary": "Packet loss incident active in Mumbai region."},
]

TELECOM_GUIDANCE = [
    {
        "doc_id": "DOC001",
        "city": "Pune",
        "issue_type": "slow_internet",
        "content": "If tower utilization exceeds 90 percent during peak hours, classify as likely congestion and notify NOC."
    },
    {
        "doc_id": "DOC002",
        "city": "Mumbai",
        "issue_type": "packet_loss",
        "content": "If packet loss affects browsing, investigate transport path degradation and escalate to network operations."
    },
    {
        "doc_id": "DOC003",
        "city": "Pune",
        "issue_type": "device_issue",
        "content": "Battery saver mode, APN settings, and network reset issues can reduce data performance."
    },
]

## Step 5 — Mock vector search (Milvus / Zilliz Cloud concept)

In [4]:
def mock_vector_search(query_text: str, city_filter: Optional[str] = None, issue_filter: Optional[str] = None, top_k: int = 2):
    q_terms = set(query_text.lower().split())
    scored = []

    for row in TELECOM_GUIDANCE:
        if city_filter and row["city"] != city_filter:
            continue
        if issue_filter and row["issue_type"] != issue_filter:
            continue

        overlap = len(q_terms & set(row["content"].lower().split()))
        scored.append({**row, "score": overlap})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

mock_vector_search("slow internet in Pune during peak hours", city_filter="Pune")

[{'doc_id': 'DOC001',
  'city': 'Pune',
  'issue_type': 'slow_internet',
  'content': 'If tower utilization exceeds 90 percent during peak hours, classify as likely congestion and notify NOC.',
  'score': 2},
 {'doc_id': 'DOC003',
  'city': 'Pune',
  'issue_type': 'device_issue',
  'content': 'Battery saver mode, APN settings, and network reset issues can reduce data performance.',
  'score': 0}]

## Step 6 — Token optimization helper

In [5]:
def build_compact_context(customer_profile: Dict[str, Any], outage_info: Dict[str, Any], guidance_rows: List[Dict[str, Any]]) -> str:
    guidance_text = " ".join([g["content"] for g in guidance_rows[:2]])
    context = (
        f"Customer city: {customer_profile['city']}. "
        f"Plan: {customer_profile['plan']}. "
        f"Outage summary: {outage_info['summary']}. "
        f"Guidance: {guidance_text}"
    )
    return context

sample_context = build_compact_context(
    CUSTOMERS["CUST_1001"],
    OUTAGES[0],
    mock_vector_search("slow internet in Pune", city_filter="Pune", issue_filter="slow_internet")
)
sample_context

'Customer city: Pune. Plan: Premium 5G. Outage summary: No citywide outage found.. Guidance: If tower utilization exceeds 90 percent during peak hours, classify as likely congestion and notify NOC.'

## Step 7 — Token and cost estimator

In [6]:
def estimate_tokens(text: str) -> int:
    return max(1, len(text.split()))

def estimate_cost_usd(prompt_tokens: int, completion_tokens: int, input_cost_per_1k=0.0005, output_cost_per_1k=0.0015):
    return round((prompt_tokens / 1000) * input_cost_per_1k + (completion_tokens / 1000) * output_cost_per_1k, 6)

estimate_cost_usd(estimate_tokens(sample_context), 60)

0.000104

## Step 8 — Model selection concept

In [7]:
MODEL_CONFIG = {
    "planner_model": "small-fast-model",
    "generator_model": "larger-accurate-model",
    "validator_model": "small-fast-model",
}
MODEL_CONFIG

{'planner_model': 'small-fast-model',
 'generator_model': 'larger-accurate-model',
 'validator_model': 'small-fast-model'}

## Step 9 — Caching

In [8]:
CACHE = {}

def cache_key(customer_id: str, issue_type: str, user_query: str) -> str:
    raw = f"{customer_id}|{issue_type}|{user_query}".encode("utf-8")
    return hashlib.md5(raw).hexdigest()

cache_key("CUST_1001", "slow_internet", "My mobile internet is slow in Pune")

'f0839681c710e5b14656a21317d3d0a7'

## Step 10 — Rate limiting

In [9]:
RATE_LIMITS = {
    "search_vector_store": 5,
    "create_ticket": 2,
}

RATE_COUNTERS = {
    "search_vector_store": 0,
    "create_ticket": 0,
}

def check_rate_limit(tool_name: str):
    RATE_COUNTERS[tool_name] += 1
    allowed = RATE_COUNTERS[tool_name] <= RATE_LIMITS[tool_name]
    return allowed, RATE_COUNTERS[tool_name]

check_rate_limit("search_vector_store"), check_rate_limit("search_vector_store")

((True, 1), (True, 2))

## Step 11 — Telecom helper tools

In [10]:
TICKETS = []

def get_customer_profile(customer_id: str) -> Dict[str, Any]:
    return CUSTOMERS[customer_id]

def check_outage(city: str, issue_type: str) -> Dict[str, Any]:
    match = next(
        (row for row in OUTAGES if row["city"] == city and row["issue_type"] == issue_type),
        {"city": city, "issue_type": issue_type, "outage_detected": False, "summary": "No outage found."}
    )
    return match

def search_vector_store(query_text: str, city_filter: Optional[str] = None, issue_filter: Optional[str] = None):
    allowed, _ = check_rate_limit("search_vector_store")
    if not allowed:
        return [{"doc_id": "RATE_LIMIT", "content": "Search rate limit reached. Use fallback guidance.", "score": 0}]
    return mock_vector_search(query_text, city_filter=city_filter, issue_filter=issue_filter)

def create_ticket(customer_id: str, city: str, issue_type: str, diagnosis: str):
    allowed, _ = check_rate_limit("create_ticket")
    if not allowed:
        return {"ticket_id": None, "status": "rate_limited", "queue": None}

    ticket = {
        "ticket_id": f"TKT-{len(TICKETS) + 1}",
        "customer_id": customer_id,
        "city": city,
        "issue_type": issue_type,
        "diagnosis": diagnosis,
        "status": "OPEN",
        "queue": "NOC",
    }
    TICKETS.append(ticket)
    return ticket

## Step 12 — Enterprise architecture idea

### Planner
Decides:
- what tools to use
- whether retrieval is needed
- whether escalation may be needed

### Executor
Calls tools:
- profile
- outage
- vector search
- ticketing

### Validator
Checks:
- is diagnosis grounded?
- is action safe?
- is response complete?

This is the **Planner – Executor – Validator** pattern.

## Step 13 — LangGraph state

In [38]:
class TelecomPerfState(TypedDict, total=False):
    customer_id: str
    issue_type: str
    user_query: str

    customer_profile: Dict[str, Any]
    outage_info: Dict[str, Any]
    guidance_results: List[Dict[str, Any]]

    plan_text: str
    diagnosis: str
    next_action: str
    validation_result: str

    cached: bool
    ticket: Dict[str, Any]

    prompt_tokens: int
    completion_tokens: int
    estimated_cost_usd: float

    final_response: str

## Step 14 — Planner node

In [39]:
def planner_node(state: TelecomPerfState) -> Dict[str, Any]:
    plan_text = (
        "1. Get customer profile. "
        "2. Check outage. "
        "3. Retrieve telecom guidance. "
        "4. Diagnose likely issue. "
        "5. Validate answer. "
        "6. Create ticket if needed."
    )
    return {"plan_text": plan_text}

## Step 15 — Executor nodes

In [40]:
def profile_node(state: TelecomPerfState) -> Dict[str, Any]:
    profile = get_customer_profile(state["customer_id"])
    return {"customer_profile": profile}

def outage_node(state: TelecomPerfState) -> Dict[str, Any]:
    city = state["customer_profile"]["city"]
    outage = check_outage(city, state["issue_type"])
    return {"outage_info": outage}

def retrieval_node(state: TelecomPerfState) -> Dict[str, Any]:
    key = cache_key(state["customer_id"], state["issue_type"], state["user_query"])

    if key in CACHE:
        cached_result = dict(CACHE[key])
        cached_result["cached"] = True
        return cached_result

    city = state["customer_profile"]["city"]
    guidance = search_vector_store(state["user_query"], city_filter=city, issue_filter=state["issue_type"])
    return {"guidance_results": guidance, "cached": False}

## Step 16 — Diagnosis node

In [41]:
def diagnosis_node(state: TelecomPerfState) -> Dict[str, Any]:
    profile = state["customer_profile"]
    outage = state["outage_info"]
    guidance = state.get("guidance_results", [])

    context = build_compact_context(profile, outage, guidance)
    prompt_tokens = estimate_tokens(context)

    if outage["outage_detected"]:
        diagnosis = f"Confirmed outage: {outage['summary']}"
        next_action = "Communicate outage status and escalate to NOC."
    elif guidance:
        diagnosis = guidance[0]["content"]
        next_action = "Follow SOP guidance and monitor the issue."
    else:
        diagnosis = "No matching guidance found."
        next_action = "Manual triage required."

    completion_tokens = estimate_tokens(diagnosis + " " + next_action)
    cost = estimate_cost_usd(prompt_tokens, completion_tokens)

    return {
        "diagnosis": diagnosis,
        "next_action": next_action,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "estimated_cost_usd": cost,
    }

## Step 17 — Validator node

In [42]:
def validator_node(state: TelecomPerfState) -> Dict[str, Any]:
    diagnosis = state["diagnosis"].lower()
    next_action = state["next_action"].lower()

    if "no matching guidance" in diagnosis:
        validation = "needs_review"
    elif "follow" in next_action or "escalate" in next_action or "communicate" in next_action:
        validation = "pass"
    else:
        validation = "needs_review"

    return {"validation_result": validation}

## Step 18 — Ticket node

In [43]:
def ticket_node(state: TelecomPerfState) -> Dict[str, Any]:
    profile = state["customer_profile"]
    diagnosis = state["diagnosis"].lower()

    if state["validation_result"] == "pass" and ("outage" in diagnosis or "notify noc" in diagnosis or "escalate" in state["next_action"].lower()):
        ticket = create_ticket(
            customer_id=profile["customer_id"],
            city=profile["city"],
            issue_type=state["issue_type"],
            diagnosis=state["diagnosis"],
        )
        return {"ticket": ticket}

    return {"ticket": {"ticket_id": None, "status": "not_created", "queue": None}}

## Step 19 — Response node

In [44]:
def response_node(state: TelecomPerfState) -> Dict[str, Any]:
    profile = state["customer_profile"]
    ticket = state["ticket"]

    response = (
        f"Customer: {profile['name']}\n"
        f"City: {profile['city']}\n"
        f"Issue Type: {state['issue_type']}\n"
        f"Plan: {profile['plan']}\n"
        f"Diagnosis: {state['diagnosis']}\n"
        f"Next Action: {state['next_action']}\n"
        f"Validation: {state['validation_result']}\n"
        f"Cached: {state.get('cached', False)}\n"
        f"Prompt Tokens: {state['prompt_tokens']}\n"
        f"Completion Tokens: {state['completion_tokens']}\n"
        f"Estimated Cost USD: {state['estimated_cost_usd']}\n"
        f"Ticket: {ticket['ticket_id']} ({ticket['status']})"
    )

    key = cache_key(state["customer_id"], state["issue_type"], state["user_query"])
    CACHE[key] = {
        "guidance_results": state.get("guidance_results", []),
        "diagnosis": state["diagnosis"],
        "next_action": state["next_action"],
        "prompt_tokens": state["prompt_tokens"],
        "completion_tokens": state["completion_tokens"],
        "estimated_cost_usd": state["estimated_cost_usd"],
    }

    return {"final_response": response}

## Step 20 — Build the LangGraph workflow

In [45]:
builder = StateGraph(TelecomPerfState)

builder.add_node("planner", planner_node)
builder.add_node("profile", profile_node)
builder.add_node("outage", outage_node)
builder.add_node("retrieval", retrieval_node)
builder.add_node("diagnosis_n", diagnosis_node)
builder.add_node("validator", validator_node)
builder.add_node("ticket_n", ticket_node)
builder.add_node("response", response_node)

builder.set_entry_point("planner")
builder.add_edge("planner", "profile")
builder.add_edge("profile", "outage")
builder.add_edge("outage", "retrieval")
builder.add_edge("retrieval", "diagnosis_n")
builder.add_edge("diagnosis_n", "validator")
builder.add_edge("validator", "ticket_n")
builder.add_edge("ticket_n", "response")
builder.add_edge("response", END)

workflow = builder.compile()
print("Workflow compiled")

Workflow compiled


## Step 21 — Run the mini workflow

In [46]:
RATE_COUNTERS["search_vector_store"] = 0
RATE_COUNTERS["create_ticket"] = 0

state = {
    "customer_id": "CUST_1001",
    "issue_type": "slow_internet",
    "user_query": "My mobile internet is slow in Pune during peak hours.",
}

result = workflow.invoke(state)
print(result["final_response"])

Customer: Ravi Patil
City: Pune
Issue Type: slow_internet
Plan: Premium 5G
Diagnosis: If tower utilization exceeds 90 percent during peak hours, classify as likely congestion and notify NOC.
Next Action: Follow SOP guidance and monitor the issue.
Validation: pass
Cached: False
Prompt Tokens: 29
Completion Tokens: 23
Estimated Cost USD: 4.9e-05
Ticket: TKT-1 (OPEN)


## Step 22 — Run again to see caching

In [47]:
RATE_COUNTERS["search_vector_store"] = 0
RATE_COUNTERS["create_ticket"] = 0

result_cached = workflow.invoke(state)
print(result_cached["final_response"])

Customer: Ravi Patil
City: Pune
Issue Type: slow_internet
Plan: Premium 5G
Diagnosis: If tower utilization exceeds 90 percent during peak hours, classify as likely congestion and notify NOC.
Next Action: Follow SOP guidance and monitor the issue.
Validation: pass
Cached: True
Prompt Tokens: 29
Completion Tokens: 23
Estimated Cost USD: 4.9e-05
Ticket: TKT-2 (OPEN)


## Step 23 — Cost comparison example

In [48]:
compact_prompt = sample_context
large_prompt = sample_context + " " + " ".join([row["content"] for row in TELECOM_GUIDANCE])

compact_tokens = estimate_tokens(compact_prompt)
large_tokens = estimate_tokens(large_prompt)

comparison = pd.DataFrame([
    {
        "scenario": "compact_context",
        "prompt_tokens": compact_tokens,
        "estimated_cost_usd": estimate_cost_usd(compact_tokens, 60),
    },
    {
        "scenario": "large_context",
        "prompt_tokens": large_tokens,
        "estimated_cost_usd": estimate_cost_usd(large_tokens, 60),
    },
])

comparison

,scenario,prompt_tokens,estimated_cost_usd
0,compact_context,29,0.000104
1,large_context,72,0.000126


## Step 24 — Rate limiting demonstration

In [49]:
RATE_COUNTERS["search_vector_store"] = 0
for i in range(7):
    rows = search_vector_store("slow internet Pune", city_filter="Pune", issue_filter="slow_internet")
    print(f"Call {i+1}: {rows[0]['doc_id']}")

Call 1: DOC001
Call 2: DOC001
Call 3: DOC001
Call 4: DOC001
Call 5: DOC001
Call 6: RATE_LIMIT
Call 7: RATE_LIMIT


## Step 25 — Enterprise architecture summary

### Planner – Executor – Validator
A useful pattern for enterprise telecom workflows.

### Multi-agent system
You can later split this into:
- planning agent
- retrieval agent
- validation agent
- escalation agent

### OSS/BSS integration
- outage tool -> OSS
- customer profile -> BSS / CRM
- ticket creation -> BSS / ITSM
- vector retrieval -> knowledge layer

## Step 26 — Where LangSmith fits

In a real deployment, LangSmith can help you:
- trace each node
- inspect token and cost trends
- debug failures
- compare workflow versions
- group traces into projects

## Step 27 — Where Zilliz Cloud fits

This notebook uses a mock vector store for simplicity.

In a real system:
- replace `mock_vector_search` with Zilliz Cloud / Milvus retrieval
- store SOPs, RCAs, incident summaries, runbooks, and troubleshooting docs
- use metadata filters for city, issue type, severity, and time

## Step 28 — Exercises

1. Add a second validator for hallucination checks
2. Add a cheaper model path for easy cases and compare cost
3. Add memory of previous complaints
4. Add a subgraph for ticket creation
5. Replace the mock vector store with real Zilliz Cloud retrieval

# Summary

This notebook demonstrated:

- performance and cost concepts
- token optimization
- model selection
- caching
- rate limiting
- cost comparison
- planner / executor / validator workflow
- enterprise architecture concepts
- multi-agent telecom workflow ideas
- OSS / BSS integration concepts
- a mini LangGraph telecom workflow

This gives you a strong beginner foundation for designing practical telecom AI systems.